<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/tensorrt_torchtrt_efficientnet/nvidia_logo.png" width="90px">

# PySpark vLLM Inference: Data Parallel + Tensor Parallel

In this notebook, we demonstrate distributed inference with the state-of-the-art "small" language model, the Mistral-Small-24B-Instruct, using open-weights on Huggingface.

The Mistral-Small-24B-Instruct is an instruction-fine-tuned version of the Mistral-Small-24B-Base model, and is competitive with larger models such as Llama 3.3 70B or Qwen 32B.

**Note:** Running this model on GPU requires ~55 GB of GPU RAM.

First visit the [Mistral Huggingface repository](https://huggingface.co/mistralai/Mistral-Small-24B-Instruct-2501) to accept the terms to access the model, then login via huggingface_hub.

In [1]:
# from huggingface_hub import login

# login()

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "14,15"
os.environ["HF_HOME"] = "/raid/spark-team/rishic/hf_home"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [2]:
from vllm import LLM, SamplingParams

# note: max_model_len needs to be <= the size of the KV cache.
# note: using tekken tokenizer (defualt) vs. mistral tokenizer for speed.
mistral_small = LLM("mistralai/Mistral-Small-24B-Instruct-2501",
                    tensor_parallel_size=2,
                    dtype="half",
                    max_model_len=23792)

# Model size: ~48GB, KV cache: ~5GB
# +-----------------------------------------+----------------------+----------------------+
# |  14  Tesla V100-SXM3-32GB-H         On  | 00000000:E5:00.0 Off |                    0 |
# | N/A   30C    P0              64W / 450W |  28047MiB / 32768MiB |      0%      Default |
# |                                         |                      |                  N/A |
# +-----------------------------------------+----------------------+----------------------+
# |  15  Tesla V100-SXM3-32GB-H         On  | 00000000:E7:00.0 Off |                    0 |
# | N/A   31C    P0              66W / 450W |  27917MiB / 32768MiB |      0%      Default |
# |                                         |                      |                  N/A |
# +-----------------------------------------+----------------------+----------------------+

INFO 02-05 23:36:26 __init__.py:183] Automatically detected platform cuda.
WARNING 02-05 23:36:28 config.py:2368] Casting torch.bfloat16 to torch.float16.
INFO 02-05 23:36:34 config.py:526] This model supports multiple tasks: {'classify', 'reward', 'embed', 'score', 'generate'}. Defaulting to 'generate'.
INFO 02-05 23:36:34 config.py:1383] Defaulting to use mp for distributed inference
INFO 02-05 23:36:34 llm_engine.py:232] Initializing a V0 LLM engine (v0.7.1) with config: model='mistralai/Mistral-Small-24B-Instruct-2501', speculative_config=None, tokenizer='mistralai/Mistral-Small-24B-Instruct-2501', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=23792, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=c

/rishic/anaconda3/envs/spark-dl-vllm/lib/python3.11/site-packages/vllm/transformers_utils/tokenizer_group/tokenizer_group.py:23: FutureWarning: It is strongly recommended to run mistral models with `--tokenizer-mode "mistral"` to ensure correct encoding and decoding.
  self.tokenizer = get_tokenizer(self.tokenizer_id, **tokenizer_config)


WARNING 02-05 23:36:35 multiproc_worker_utils.py:298] Reducing Torch parallelism from 48 threads to 1 to avoid unnecessary CPU contention. Set OMP_NUM_THREADS in the external environment to tune this value as needed.
INFO 02-05 23:36:35 custom_cache_manager.py:17] Setting Triton cache manager to: vllm.triton_utils.custom_cache_manager:CustomCacheManager
(VllmWorkerProcess pid=2869092) INFO 02-05 23:36:35 multiproc_worker_utils.py:227] Worker ready; awaiting tasks
INFO 02-05 23:36:36 cuda.py:184] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 02-05 23:36:36 cuda.py:232] Using XFormers backend.
(VllmWorkerProcess pid=2869092) INFO 02-05 23:36:36 cuda.py:184] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
(VllmWorkerProcess pid=2869092) INFO 02-05 23:36:36 cuda.py:232] Using XFormers backend.
INFO 02-05 23:36:37 utils.py:938] Found nccl from library libnccl.so.2
INFO 02-05 23:36:37 pynccl.py:67] vLLM is using nccl==2.21.5
(VllmWorkerProcess pid=2869092

Loading safetensors checkpoint shards:   0% Completed | 0/10 [00:00<?, ?it/s]


INFO 02-05 23:37:10 model_runner.py:1116] Loading model weights took 21.9618 GB
(VllmWorkerProcess pid=2869092) INFO 02-05 23:37:10 model_runner.py:1116] Loading model weights took 21.9618 GB
INFO 02-05 23:37:19 worker.py:266] Memory profiling takes 8.29 seconds
INFO 02-05 23:37:19 worker.py:266] the current vLLM instance can use total_gpu_memory (31.74GiB) x gpu_memory_utilization (0.90) = 28.57GiB
INFO 02-05 23:37:19 worker.py:266] model weights take 21.96GiB; non_torch_memory takes 0.84GiB; PyTorch activation peak memory takes 2.87GiB; the rest of the memory reserved for KV Cache is 2.89GiB.
(VllmWorkerProcess pid=2869092) INFO 02-05 23:37:19 worker.py:266] Memory profiling takes 8.17 seconds
(VllmWorkerProcess pid=2869092) INFO 02-05 23:37:19 worker.py:266] the current vLLM instance can use total_gpu_memory (31.74GiB) x gpu_memory_utilization (0.90) = 28.57GiB
(VllmWorkerProcess pid=2869092) INFO 02-05 23:37:19 worker.py:266] model weights take 21.96GiB; non_torch_memory takes 0.71

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

(VllmWorkerProcess pid=2869092) INFO 02-05 23:37:22 model_runner.py:1435] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_utilization` or switching to eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.


Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:22<00:00,  1.56it/s]

INFO 02-05 23:37:45 custom_all_reduce.py:224] Registering 2835 cuda graph addresses


(VllmWorkerProcess pid=2869092) INFO 02-05 23:37:46 custom_all_reduce.py:224] Registering 2835 cuda graph addresses
(VllmWorkerProcess pid=2869092) INFO 02-05 23:37:46 model_runner.py:1563] Graph capturing finished in 24 secs, took 1.27 GiB
INFO 02-05 23:37:46 model_runner.py:1563] Graph capturing finished in 24 secs, took 1.27 GiB
INFO 02-05 23:37:46 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 36.07 seconds


In [7]:
prompts = ["How many r's are in the word 'strawberry'?",
           "Which number is bigger: 9.9 or 9.11?",
           "Can you draw an ASCII cat?"]

sampling_params = SamplingParams(temperature=0.15, max_tokens=512)
outputs = mistral_small.generate(prompts, sampling_params)

Processed prompts: 100%|██████████| 3/3 [00:06<00:00,  2.21s/it, est. speed input: 6.05 toks/s, output: 46.85 toks/s] 


In [8]:
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print("")
    print(f"----> Prompt: {prompt}\n")
    print(f"Generated text: {generated_text}\n")


----> Prompt: How many r's are in the word 'strawberry'?

Generated text:  There are 4 r's in the word 'strawberry'.


----> Prompt: Which number is bigger: 9.9 or 9.11?

Generated text:  To determine which number is larger, we need to compare the digits in each place value.

1. **Compare the whole number parts**:
   - Both numbers have a whole number part of 9.

2. **Compare the tenths place**:
   - 9.9 has 9 in the tenths place.
   - 9.11 has 1 in the tenths place.

Since 9 is greater than 1, 9.9 is greater than 9.11.

Therefore, the bigger number is $\boxed{9.9}$.


----> Prompt: Can you draw an ASCII cat?

Generated text:  Here's a simple one:

```
 /\_/\
( o.o )
 > ^ <
```

This is a fun way to practice drawing with text. You can create more complex designs by using different characters and spacing. For example, you can add more details to the cat's face or body. Here's a slightly more detailed version:

```
 /\_/\
( o.o )
 > ^ <
```

You can also experiment with different styles

In [9]:
import torch
del mistral_small
torch.cuda.empty_cache()

INFO 02-05 23:39:25 multiproc_worker_utils.py:139] Terminating local vLLM worker processes


(VllmWorkerProcess pid=2869092) INFO 02-05 23:39:25 multiproc_worker_utils.py:251] Worker exiting


## PySpark

In [1]:
from pyspark.sql.types import *
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col, struct, length
from pyspark.ml.functions import predict_batch_udf

In [2]:
import os

os.environ["HF_HOME"] = "/raid/spark-team/rishic/hf_home"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [3]:
import datasets
from datasets import load_dataset
datasets.disable_progress_bars()

#### Create Spark Session

For local standalone clusters, we'll connect to the cluster and create the Spark Session.  
For CSP environments, Spark will either be preconfigured (Databricks) or we'll need to create the Spark Session (Dataproc).

In [4]:
import socket
conda_env = os.environ.get("CONDA_PREFIX")
hostname = socket.gethostname()

conf = SparkConf()
conf.setMaster(f"spark://{hostname}:7077")
conf.set("spark.pyspark.python", f"{conda_env}/bin/python")
conf.set("spark.pyspark.driver.python", f"{conda_env}/bin/python")
conf.set("spark.executorEnv.HF_HOME", "/raid/spark-team/rishic/hf_home")  # Executors can load model from cache
conf.set("spark.executor.instances", "2")
conf.set("spark.executorEnv.LD_LIBRARY_PATH", f"{conda_env}/lib:{conda_env}/lib/python3.11/site-packages/nvidia_pytriton.libs:$LD_LIBRARY_PATH")
conf.set("spark.executor.cores", "16")
conf.set("spark.task.maxFailures", "1")
conf.set("spark.task.resource.gpu.amount", "0.125")  # Allow 16 inference request tasks to run in parallel
conf.set("spark.executor.resource.gpu.amount", "2")
conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
conf.set("spark.python.worker.reuse", "true")

spark = SparkSession.builder.appName("spark-dl-examples").config(conf=conf).getOrCreate()
sc = spark.sparkContext

25/02/05 23:43:46 WARN Utils: Your hostname, dgx2h0194.spark.sjc4.nvmetal.net resolves to a loopback address: 127.0.1.1; using 10.150.30.2 instead (on interface enp134s0f0np0)
25/02/05 23:43:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/05 23:43:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
dataset = load_dataset("Open-Orca/OpenOrca", split="train[:1%]")
dataset = dataset.to_pandas()["question"]

In [6]:
df = spark.createDataFrame(dataset, schema=StringType()).withColumnRenamed("value", "prompt")
df.show(5, truncate=100)

+----------------------------------------------------------------------------------------------------+
|                                                                                              prompt|
+----------------------------------------------------------------------------------------------------+
|You will be given a definition of a task first, then some input of the task.\nThis task is about ...|
|Generate an approximately fifteen-word sentence that describes all this data: Midsummer House eat...|
|What happens next in this paragraph?\n\nShe then rubs a needle on a cotton ball then pushing it o...|
|Please answer the following question: I want to test the ability of students to read a passage an...|
|James runs a TV show and there are 5 main characters and 4 minor characters. He pays the minor ch...|
+----------------------------------------------------------------------------------------------------+
only showing top 5 rows



In [7]:
df.filter(length(col("prompt")) <= 100).limit(100).write.mode("overwrite").json("spark-dl-datasets/open_orca")

25/02/05 23:43:54 WARN TaskSetManager: Stage 1 contains a task of very large size (1001 KiB). The maximum recommended task size is 1000 KiB.


In [8]:
df = spark.read.json("spark-dl-datasets/open_orca")

In [9]:
# demo with shorter prompts
df = df.limit(256).repartition(32).cache()

In [10]:
df.show(truncate=False)

+---------------------------------------------------------------------------------------------------+
|prompt                                                                                             |
+---------------------------------------------------------------------------------------------------+
|What are the keywords in the following sentence:\n\nflowers blooming where the time seems to stop .|
|How is "No, just waiting for Erica." said in Czech?                                                |
|Generate a negative review for a place.                                                            |
|Short general knowledge question: where is st helens park nsw?\nA:                                 |
|Please add spaces between words: PalermoCathedral-pictures,Italy,photosofItaly,Italypictures       |
|Answer the question...when was the flight of the bumblebee written??                               |
|I went dependent on insulin in 1996.\n\nPlease remove spaces between words.      

## Triton Inference Server

In [11]:
import json
from functools import partial

Define the Triton Server function:

In [35]:
def triton_server(ports):
    import time
    import signal
    import numpy as np
    from pytriton.decorators import batch
    from pytriton.model_config import DynamicBatcher, ModelConfig, Tensor
    from pytriton.triton import Triton, TritonConfig
    from pyspark import TaskContext
    from vllm import LLM, SamplingParams

    print(f"SERVER: Initializing model on worker {TaskContext.get().partitionId()}.")
    print(f"SERVER: Using HF cache: {os.environ.get('HF_HOME')}")

    if TaskContext.get().resources().get("gpu").addresses[0] in ["14", "15"]:
        os.environ["CUDA_VISIBLE_DEVICES"] = "14,15"
    else:
        os.environ["CUDA_VISIBLE_DEVICES"] = "12,13"
    
    print("Using GPUs:", os.environ["CUDA_VISIBLE_DEVICES"])
    
    mistral_small = LLM("mistralai/Mistral-Small-24B-Instruct-2501",
                        tensor_parallel_size=2,
                        dtype="half",
                        max_model_len=23792)
    sampling_params = SamplingParams(temperature=0.15, max_tokens=256)

    @batch
    def _infer_fn(**inputs):
        prompts = np.squeeze(inputs["prompts"]).tolist()
        decoded_prompts = [p.decode("utf-8") for p in prompts]
        responses = mistral_small.generate(decoded_prompts, sampling_params)
        return {
            "responses": np.array([r.outputs[0].text for r in responses]).reshape(-1, 1),
        }

    workspace_path = f"/tmp/triton_{TaskContext.get().partitionId()}_{time.strftime('%m_%d_%M_%S')}"
    triton_conf = TritonConfig(http_port=ports[0], grpc_port=ports[1], metrics_port=ports[2])
    with Triton(config=triton_conf, workspace=workspace_path) as triton:
        triton.bind(
            model_name="mistral-small-3",
            infer_func=_infer_fn,
            inputs=[
                Tensor(name="prompts", dtype=object, shape=(-1,)),
            ],
            outputs=[
                Tensor(name="responses", dtype=object, shape=(-1,)),
            ],
            config=ModelConfig(
                max_batch_size=16,
                batcher=DynamicBatcher(max_queue_delay_microseconds=5000),  # 5ms
            ),
            strict=True,
        )

        def _stop_triton(signum, frame):
            print("SERVER: Received SIGTERM. Stopping Triton server.")
            triton.stop()

        signal.signal(signal.SIGTERM, _stop_triton)

        print("SERVER: Serving inference")
        triton.serve()

In [36]:
def start_triton(triton_server_fn, ports, model_name):
    from pytriton.client import ModelClient
    from multiprocessing import Process

    ports = next(ports)

    hostname = socket.gethostname()
    process = Process(target=triton_server_fn, args=(ports,))
    process.start()

    client = ModelClient(f"http://localhost:{ports[0]}", model_name)
    patience = 30
    for _ in range(patience):
        try:
            client.wait_for_model(5)
            return [(hostname, process.pid)]
        except Exception:
            print("Waiting for server to be ready...")

    emsg = "Failure: client waited too long for server startup. Check the executor logs for more info."
    raise TimeoutError(emsg)

#### Start Triton servers

**Specify the number of nodes in the cluster.**  
Following the README, the example standalone cluster uses 1 node. The example Databricks/Dataproc cluster scripts use 4 nodes by default. 

In [37]:
# Change based on cluster setup
num_nodes = 2

To ensure that only one Triton inference server is started per node, we use stage-level scheduling to delegate each task to a separate GPU.  

In [38]:
def use_stage_level_scheduling(rdd):

    from pyspark.resource.profile import ResourceProfileBuilder
    from pyspark.resource.requests import TaskResourceRequests

    executor_cores = spark.conf.get("spark.executor.cores")

    task_cores = (int(executor_cores) // 2) + 1
    task_gpus = 2.0
    
    treqs = TaskResourceRequests().cpus(task_cores).resource("gpu", task_gpus)
    rp = ResourceProfileBuilder().require(treqs).build
    print(f"Requesting stage-level resources: (cores={task_cores}, gpu={task_gpus})")

    return rdd.withResources(rp)

In [39]:
sc = spark.sparkContext
nodeRDD = sc.parallelize([(7000, 7001, 7002), (7003, 7004, 7005)], num_nodes)
nodeRDD = use_stage_level_scheduling(nodeRDD)

Requesting stage-level resources: (cores=9, gpu=2.0)


In [40]:
model_name = "mistral-small-3"

In [41]:
pids = nodeRDD.barrier().mapPartitions(lambda ports: start_triton(triton_server_fn=triton_server, ports=ports, model_name=model_name)).collect()
print("Triton Server PIDs:\n", pids)

Triton Server PIDs:
 [('dgx2h0194.spark.sjc4.nvmetal.net', 2886044), ('dgx2h0194.spark.sjc4.nvmetal.net', 2886045)]


In [47]:
# Server 1: GPUs [12, 13]
# Server 2: GPUs [14, 15]
# +-----------------------------------------+----------------------+----------------------+
# |  12  Tesla V100-SXM3-32GB-H         On  | 00000000:E0:00.0 Off |                    0 |
# | N/A   37C    P0             205W / 450W |  28549MiB / 32768MiB |     96%      Default |
# |                                         |                      |                  N/A |
# +-----------------------------------------+----------------------+----------------------+
# |  13  Tesla V100-SXM3-32GB-H         On  | 00000000:E2:00.0 Off |                    0 |
# | N/A   40C    P0             231W / 450W |  28389MiB / 32768MiB |     95%      Default |
# |                                         |                      |                  N/A |
# +-----------------------------------------+----------------------+----------------------+
# |  14  Tesla V100-SXM3-32GB-H         On  | 00000000:E5:00.0 Off |                    0 |
# | N/A   42C    P0             245W / 450W |  28551MiB / 32768MiB |     96%      Default |
# |                                         |                      |                  N/A |
# +-----------------------------------------+----------------------+----------------------+
# |  15  Tesla V100-SXM3-32GB-H         On  | 00000000:E7:00.0 Off |                    0 |
# | N/A   44C    P0             244W / 450W |  28391MiB / 32768MiB |     95%      Default |
# |                                         |                      |                  N/A |
# +-----------------------------------------+----------------------+----------------------+

#### Define client function

In [42]:
e0_url = f"grpc://localhost:7001"
e1_url = f"grpc://localhost:7004"

In [43]:
def triton_fn(model_name):
    import numpy as np
    from pytriton.client import ModelClient
    from pyspark import TaskContext

    if TaskContext.get().resources().get("gpu").addresses[0] in ["14", "15"]:
        url = e0_url
    else:
        url = e1_url
        
    print(f"Connecting to Triton model {model_name} at {url}.")

    def infer_batch(inputs):
        with ModelClient(url, model_name, inference_timeout_s=500) as client:
            flattened = np.squeeze(inputs).tolist()
            # Encode batch
            encoded_batch = [[text.encode("utf-8")] for text in flattened]
            encoded_batch_np = np.array(encoded_batch, dtype=np.bytes_)
            # Run inference
            result_data = client.infer_batch(encoded_batch_np)
            result_data = np.squeeze(result_data["responses"], -1)
            return result_data
        
    return infer_batch

#### Load DataFrame

In [44]:
df.show(truncate=100)

+---------------------------------------------------------------------------------------------------+
|                                                                                             prompt|
+---------------------------------------------------------------------------------------------------+
|What are the keywords in the following sentence:\n\nflowers blooming where the time seems to stop .|
|                                                How is "No, just waiting for Erica." said in Czech?|
|                                                            Generate a negative review for a place.|
|                                 Short general knowledge question: where is st helens park nsw?\nA:|
|       Please add spaces between words: PalermoCathedral-pictures,Italy,photosofItaly,Italypictures|
|                               Answer the question...when was the flight of the bumblebee written??|
|                        I went dependent on insulin in 1996.\n\nPlease remove spa

#### Run Inference

In [45]:
generate = predict_batch_udf(partial(triton_fn, model_name=model_name),
                             return_type=StringType(),
                             input_tensor_shapes=[[1]],
                             batch_size=8)

In [46]:
%%time
# first pass caches model/fn
preds = df.withColumn("response", generate(col("prompt")))
results = preds.collect()

CPU times: user 20.2 ms, sys: 1.19 ms, total: 21.4 ms
Wall time: 52.8 s


In [48]:
print(f"Q: {results[5].prompt} \n")
print(f"A: {results[5].response} \n")

Q: Answer the question...when was the flight of the bumblebee written?? 

A: ?

The "Flight of the Bumblebee" is a famous orchestral interlude written by Nikolai Rimsky-Korsakov. It was composed in 1900 as part of his opera "The Tale of Tsar Saltan." The piece is known for its rapid tempo and technical difficulty, making it a popular showpiece for musicians. 



#### Shut down server on each executor

In [24]:
shutdownRDD = sc.parallelize(list(range(num_nodes)), num_nodes)
shutdownRDD = use_stage_level_scheduling(spark, shutdownRDD)
shutdownRDD.barrier().mapPartitions(lambda _: stop_triton(pids)).collect()

Reqesting stage-level resources: (cores=5, gpu=1.0)


[True]

In [25]:
if not on_databricks: # on databricks, spark.stop() puts the cluster in a bad state
    spark.stop()